#### 5.1 统计在某一个查询中，每个post在最后结果出现次数的估计w(x)，以及每次查询的总估计W：
post_with_estimate.csv文件，统计 estimate__xxxx.graph列>0值的元组有多少count，并依次输出每列的总和sum，按每列总和排序；
设置参数count_min和sum_max，只输出count>=count_min且sum<=sum_max的查询

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
import os


def summarize_estimates(csv_file):
    """
    汇总 post_with_estimate.csv 中每个 estimate__xxx.graph 列：
    - 统计 >0 的记录数
    - 计算该列的预测值总和
    输出字典格式：{列名: (非零数量, 总和)}
    """
    if not os.path.exists(csv_file):
        raise FileNotFoundError(f"未找到文件: {csv_file}")

    df = pd.read_csv(csv_file)

    # 筛选出所有 Fastest 预测列
    estimate_cols = [c for c in df.columns if c.startswith("estimate__") and c.endswith(".graph")]

    print(f"✅ 共发现 {len(estimate_cols)} 个 estimate 列")
    print("-" * 60)

    results = {}

    for col in estimate_cols:
        # 转为 float 避免字符串干扰
        vals = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

        count_pos = (vals > 0).sum()
        total_sum = float(vals[vals > 0].sum())

        results[col] = (int(count_pos), total_sum)
        print(f"{col}: 非零数={count_pos}, 总和={total_sum:.2f}")

    print("-" * 60)
    print("最终结果字典：")
    print(results)
    return results


# === 调用示例 ===
dataset_name = 'dataset_four'
csv_path = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/results/post_with_estimate.csv"

summarize_estimates(csv_path)


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
import os


def summarize_estimates(csv_file, count_min=0, sum_max=float("inf")):
    """
    汇总 post_with_estimate.csv 中每个 estimate__xxx.graph 列：
    - 统计 >0 的记录数 count
    - 计算非零值总和 sum
    - 按总和排序（从大到小）
    - 筛选：count >= count_min 且 sum <= sum_max
    """

    if not os.path.exists(csv_file):
        raise FileNotFoundError(f"未找到文件: {csv_file}")

    df = pd.read_csv(csv_file)

    # 找出所有 estimate 列
    estimate_cols = [
        c for c in df.columns
        if c.startswith("estimate__") and c.endswith(".graph")
    ]

    print(f"✅ 共发现 {len(estimate_cols)} 个 estimate 列")
    print("-" * 70)

    results = {}

    for col in estimate_cols:
        vals = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

        count_pos = int((vals > 0).sum())
        total_sum = float(vals[vals > 0].sum())

        results[col] = (count_pos, total_sum)

    # ✅ 按 sum（第二字段）从大到小排序
    sorted_items = sorted(results.items(), key=lambda x: x[1][1], reverse=True)

    print("\n📌 排序后所有列（sum 从大到小）:")
    print("-" * 70)
    for col, (count_pos, total_sum) in sorted_items:
        print(f"{col}: count={count_pos}, sum={total_sum}")

    # ✅ 应用过滤条件
    filtered = [
        (col, (count_pos, total_sum))
        for col, (count_pos, total_sum) in sorted_items
        if count_pos >= count_min and total_sum <= sum_max
    ]

    print("\n🎯 满足过滤条件 (count >= {} 且 sum <= {} )：".format(count_min, sum_max))
    print("-" * 70)
    for col, (count_pos, total_sum) in filtered:
        print(f"{col}: count={count_pos}, sum={total_sum}")

    print("-" * 70)
    print("最终过滤后的结果列表：")
    print(filtered)
    print(len(filtered), "列符合条件。")
    return filtered, results


# === 调用示例 ===
dataset_name = "dataset_four"
csv_path = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/results/post_with_estimate.csv"

# 示例：输出 count≥1000 且 sum≤5,000,000 的列
ok, filtered_results  = summarize_estimates(csv_path, count_min=500, sum_max=100976351.54)


#### 5.2 根据w(x)和W的统计结果，移动查询图文件：
已经成功完成上面任务，每个列的格式类似下面estimate__104__query_path_8_8.graph，，它对应的查询图为在/home/wangshuo/resource/datasets/parler_data/dataset_two/query_graph 文件夹下的 query_path_8_8.graph ，我想将这个文件夹下不满足count≥1000 且 sum≤5,000,000 的 查询移动到 /home/wangshuo/resource/datasets/parler_data/dataset_two/difficult_query_graph下面去

In [ ]:
import shutil
def move_difficult_queries(results,
                           query_graph_dir,
                           difficult_dir,
                           count_min=0,
                           sum_max=float("inf")):
    """
    根据 summarize_estimates() 的结果，移动不满足条件的查询图文件。

    参数：
        results: summarize_estimates() 返回的 {col: (count, sum)}
        count_min: 允许的最小 count
        sum_max: 允许的最大 sum

    功能：
        - 满足条件的保留
        - 不满足 count>=count_min 且 sum<=sum_max 的移动到 difficult_dir
    """

    os.makedirs(difficult_dir, exist_ok=True)

    ok_queries = set()
    bad_queries = set()

    for col, (count_pos, total_sum) in results.items():

        # 提取真实图文件名
        # 如：estimate__104__query_path_8_8.graph → query_path_8_8.graph
        query_file = col.split("__")[-1]

        if count_pos >= count_min and total_sum <= sum_max:
            ok_queries.add(query_file)
        else:
            bad_queries.add(query_file)

    print(f"\n🎯 满足条件的查询: {len(ok_queries)}")
    print(f"❌ 需移动的困难查询: {len(bad_queries)}")

    moved = 0
    for qfile in bad_queries:
        src = os.path.join(query_graph_dir, qfile)
        dst = os.path.join(difficult_dir, qfile)

        if os.path.exists(src):
            shutil.move(src, dst)
            moved += 1
            print(f"📦 移动: {qfile}")
        else:
            print(f"⚠️ 未找到文件（跳过）: {qfile}")

    print(f"\n✅ 已移动 {moved} 个困难查询图文件")

    return ok_queries, bad_queries

dataset_name = 'dataset_four'
query_graph_dir = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/query_graph"
difficult_dir = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/difficult_query_graph"
# ✅ 再根据统计结果移动文件
move_difficult_queries(
    results=filtered_results,    # 这里要传所有结果
    query_graph_dir=query_graph_dir,
    difficult_dir=difficult_dir,
    count_min=1000,
    sum_max=100976351.54
)

#### 4. 随机生成parler_ans.txt文件 ，供fastest初步估计该查询的基数大小：
读取/home/wangshuo/resource/datasets/parler_data/dataset_two/query_graph 文件夹下的所有查询文件，生成以下格式的parler_ans.txt文件，时间和数值随机生成（第二列第三列随机生成，第一列是查询名字）
query_dense_1_1.graph 231.3ms 129165
query_dense_1_2.graph 157.4ms 24794
query_dense_1_3.graph 7370.7ms 3789334
query_path_4_0.graph 5893.9ms 4079965
query_path_4_2.graph 4412.0ms 2872354
.........

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import random

# 输入与输出路径
dataset_name = 'dataset_one'
query_dir = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/query_graph"
output_file = os.path.join(f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/ground_truth", "parler_ans.txt")

# 获取所有查询文件
query_files = sorted(f for f in os.listdir(query_dir) if f.endswith(".graph"))

with open(output_file, "w") as f_out:
    for qf in query_files:
        # 随机生成时间(ms)和数值
        time_ms = round(random.uniform(100.0, 150000.0), 1)
        value = random.randint(10000, 800000000)
        # 写入结果
        f_out.write(f"{qf} {time_ms}ms {value}\n")

print(f"[INFO] 已生成 {len(query_files)} 条记录到 {output_file}")


[INFO] 已生成 16 条记录到 /home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/parler_ans.txt


#### 3.新旧生成的查询图合并：
我之前通过上面代码生成过path,tree结构的查询图，保存到/home/wangshuo/resource/datasets/parler_data/dataset_one/query_graph，并执行力下游的任务，接下来我又运行了代码产生了新的查询图，保存到了/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/second_genegrate 文件夹，该文件夹下的文件有cycle_4  cycle_6  cycle_8  path_4  path_6  path_8  star_4  star_6  star_8  tree_4  tree_6  tree_8，依次读取这些文件夹下的所有查询文件，并和原有在/home/wangshuo/resource/datasets/parler_data/dataset_one/query_graph 比较，只保留和现有查询不同的查询，/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/second_genegrate 文件夹的各种查询结构文件夹下的查询图文件的编号都是从0开始的，例如 tree_4 下的查询文件如下
~/resource/datasets/parler_data/proto_query_graphs/second_genegrate/tree _4$ ls 
query_tree_4_0.graph   query_tree_4_11.graph  query_tree_4_13.graph  query_tree_4_15.graph  query_tree_4_17.graph  query_tree_4_19.graph  query_tree_4_2.graph  query_tree_4_4.graph  query_tree_4_6.graph  query_tree_4_8.graph
query_tree_4_10.graph  query_tree_4_12.graph  query_tree_4_14.graph  query_tree_4_16.graph  query_tree_4_18.graph  query_tree_4_1.graph   query_tree_4_3.graph  query_tree_4_5.graph  query_tree_4_7.graph  query_tree_4_9.graph
，然而在/home/wangshuo/resource/datasets/parler_data/dataset_one/query_graph  下如果已经有 query_tree_4_3.graph ，
如果合并的话，得从 query_tree_4_4.graph 开始编号

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
merge_new_queries_flat_to_dataset_one.py
将 second_generate/ 下的所有查询图去重后合并到 dataset_one/query_graph。
合并后所有结构（tree、path、star、cycle等）全部平铺在一个文件夹中。
"""

import os
import shutil
import filecmp

# === 路径配置 ===
OLD_DIR = "/home/wangshuo/resource/datasets/parler_data/dataset_two/query_graph"
NEW_BASE = "/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/second_genegrate"

def get_graph_files(folder):
    """返回某文件夹下所有 .graph 文件路径"""
    return sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.endswith(".graph")
    ])

def file_content_equal(f1, f2):
    """逐字节比较两个文件是否完全相同"""
    try:
        return filecmp.cmp(f1, f2, shallow=False)
    except Exception:
        return False

def get_next_index(existing_files, prefix):
    """根据已有文件名提取最大编号+1"""
    max_idx = -1
    for f in existing_files:
        base = os.path.basename(f)
        if base.startswith(prefix) and base.endswith(".graph"):
            try:
                idx = int(base.split("_")[-1].split(".")[0])
                max_idx = max(max_idx, idx)
            except ValueError:
                pass
    return max_idx + 1

def merge_structure(struct_folder):
    """处理单个结构目录（例如 tree_4, path_8 等）"""
    new_dir = os.path.join(NEW_BASE, struct_folder)
    if not os.path.exists(new_dir):
        print(f"[SKIP] {new_dir} 不存在，跳过。")
        return

    new_files = get_graph_files(new_dir)
    if not new_files:
        print(f"[SKIP] {struct_folder} 下没有 .graph 文件。")
        return

    os.makedirs(OLD_DIR, exist_ok=True)
    old_files = get_graph_files(OLD_DIR)

    struct_type, struct_size = struct_folder.split("_")
    prefix = f"query_{struct_type}_{struct_size}"

    next_idx = get_next_index(old_files, prefix)
    added = 0

    print(f"[INFO] 合并 {struct_folder} ：旧 {len(old_files)} 个，新 {len(new_files)} 个")

    for nf in new_files:
        # 检查是否与已有文件重复
        is_duplicate = any(file_content_equal(nf, of) for of in old_files)
        if not is_duplicate:
            new_name = f"{prefix}_{next_idx}.graph"
            dst_path = os.path.join(OLD_DIR, new_name)
            shutil.copy2(nf, dst_path)
            next_idx += 1
            added += 1

    print(f"[OK] {struct_folder} 合并完成：新增 {added} 个查询。")

def main():
    structs = sorted([d for d in os.listdir(NEW_BASE)
                      if os.path.isdir(os.path.join(NEW_BASE, d))])
    print(f"[INFO] 检测到结构目录 {len(structs)} 个：{structs}")
    for s in structs:
        merge_structure(s)
    print(f"[DONE] 所有查询已合并至：{OLD_DIR}")

if __name__ == "__main__":
    main()


#### 2. 生成推理节点文件 infer_node.txt：
一个查询图的格式如下，三个顶点按照id依次编号为u0,u1,u2,接下来我想读取/home/wangshuo/resource/datasets/parler_data/dataset_one/query_graph文件夹下的所有查询文件，将标签为1的顶点所对应的标号保存到infer_node.txt，以下面查询为例要保存u1到文件中，
                                            t 3 3
                                            v 0 0 2
                                            v 1 1 2
                                            v 2 2 2
                                            e 1 2
                                            e 2 0
                                            e 1 0

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
extract_infer_node.py
从 query_graph 文件夹下的所有查询 .graph 文件中，
提取 label==1 的顶点编号，写入 infer_node.txt。
"""

import os

# === 配置路径 ===
dataset_name = 'dataset_four'
query_dir = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/query_graph"
out_path = os.path.join(f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/data_graph", "infer_node.txt")

# === 收集所有 .graph 文件 ===
graph_files = []
for root, _, files in os.walk(query_dir):
    for f in files:
        if f.endswith(".graph"):
            graph_files.append(os.path.join(root, f))
graph_files.sort()

print(f"[INFO] Found {len(graph_files)} query graph files under {query_dir}")

# === 解析每个文件，提取 label==1 的节点编号 ===
results = []
for path in graph_files:
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line or not line.startswith("v "):
                continue
            parts = line.split()
            if len(parts) >= 3:
                vid = parts[1]
                label = parts[2]
                if label == "1":
                    results.append(f"u{vid}")
                    break  # 每个查询只有一个 label==1，提取后就可以退出

# === 写入输出文件 ===
with open(out_path, "w") as fout:
    for r in results:
        fout.write(r + "\n")

print(f"[DONE] Saved {len(results)} infer nodes to {out_path}")


1.2
 EDGE_LABELS={
        "author_user_post": 0,      # user -> post (creator)
        "creator_user_comment": 1,  # user -> comment (creator)
        "post_has_comment": 2,      # post -> comment (post)
        "comment_reply_comment": 3, # comment -> comment (parent)
        "user_view_post": 4         # user -> post (view)
    }

我的数据图我顶点有标签（0，1，2），但是边没有了标签（只是将上面五种边的标签去掉，且将边标签为0和4的边合并了），我想让你基于数据图生成稀疏查询和稠密查询，这两个类型分别生成数量为4，6，8的查询，每个数量的查询为N=25个，其中标签为1的节点，在每个查询图中只能出现一次（也就是数量为1）

#### 1.1 根据要求生成各种结构不同的查询星型、路径、环、树：
    EDGE_LABELS={
        "author_user_post": 0,      # user -> post (creator)
        "creator_user_comment": 1,  # user -> comment (creator)
        "post_has_comment": 2,      # post -> comment (post)
        "comment_reply_comment": 3, # comment -> comment (parent)
        "user_view_post": 4         # user -> post (view)
    }

我的数据图我顶点有标签（0，1，2），但是边没有了标签（只是将上面五种边的标签去掉，且将边标签为0和4的边合并了），我想让你基于数据图生成各种结构不同的查询星型、路径、环、树，针对每个不同的结构生成节点数量为4，6，8，12的查询，每个数量的查询为N=5个，其中标签为1的节点，再每个查询图中只能出现一次（也就是数量为1）


在生成查询的同时要保证用户指定的某一个标签l的节点在查询图中出现少于k次（可以出现1，2，3，4，5。。。k,但k越大生成的概率越小）

V2.0

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
generate_queries.py

从带节点标签（0/1/2）、无边标签的 Fastest 格式图（.graph）自动生成查询 workload。
约束：
 - 结构类型： star, path, cycle, tree
 - 节点数： 4, 6, 8, 12
 - 每种结构 x 每种节点数 至少生成 N 个（可配置）
 - 每个查询中恰好包含 1 个 label==1 的节点（Post）
 - 输出为 Fastest .graph（节点 0..n-1, 每行 v, 每行 e）
"""

import os
import random
from collections import deque
from typing import List, Set
import networkx as nx

random.seed(42)


def load_graph_fastest(filepath: str) -> nx.Graph:
    """
    读取 Fastest .graph（或类似）格式的图。
    期望 v 行： v <vid> <label> [deg]
            e 行： e <u> <v> [label?]
    只读取节点标签和无向边。
    """
    G = nx.Graph()
    if not os.path.exists(filepath):
        raise FileNotFoundError(filepath)
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if parts[0] == "v" and len(parts) >= 3:
                vid = int(parts[1])
                label = int(parts[2])
                G.add_node(vid, label=label)
            elif parts[0] == "e" and len(parts) >= 3:
                u = int(parts[1]); v = int(parts[2])
                G.add_edge(u, v)
    return G


class QueryGenerator:
    def __init__(self, G: nx.Graph, out_dir: str = "query_graphs", seed: int = 42):
        self.G = G
        self.out_dir = out_dir
        os.makedirs(self.out_dir, exist_ok=True)
        self.nodes_by_label = {}
        for n, d in self.G.nodes(data=True):
            lab = d.get("label", None)
            self.nodes_by_label.setdefault(lab, []).append(n)
        random.seed(seed)

    def _write_fastest_query(self, qnodes: List[int], edges: List[tuple], outpath: str):
        """
        将 query 写成 Fastest .graph：v idx label deg; e i j
        节点重新编号为 0..k-1（顺序按 qnodes 列表）
        """
        node_index = {n: i for i, n in enumerate(qnodes)}
        # === 计算节点在查询图中的度数 ===
        deg = {n: 0 for n in qnodes}
        for (u, v) in edges:
            deg[u] += 1
            deg[v] += 1

        with open(outpath, "w") as fout:
            fout.write(f"t {len(qnodes)} {len(edges)}\n")
            for n in qnodes:
                i = node_index[n]
                lab = int(self.G.nodes[n].get("label", 0))
                d = deg[n]
                fout.write(f"v {i} {lab} {d}\n")   # ✅ 输出度数
            for (u, v) in edges:
                fout.write(f"e {node_index[u]} {node_index[v]}\n")


    def _induced_subgraph_edges(self, nodes: List[int]):
        """返回节点集的无向边（列表形式）"""
        edges = []
        s = set(nodes)
        for u in nodes:
            for v in self.G.neighbors(u):
                if v in s and u < v:
                    edges.append((u, v))
        return edges

    def _count_label1(self, nodes: List[int]) -> int:
        # 改标签替换这里
        return sum(1 for n in nodes if int(self.G.nodes[n].get("label", 0)) == 2)

    def _ensure_exact_one_label1(self, nodes: List[int]) -> bool:
        """检查节点集合是否恰好含 1 个 label==1"""
        return self._count_label1(nodes) == 1

    # ------------------- 结构生成器 -------------------

    def gen_star_one(self, size: int, max_attempts: int = 500) -> List[int]:
        """
        尝试生成 size 个节点的 star-like connected set:
        - pick a center node c, pick size-1 neighbors (if not enough neighbors, expand by 2-hop)
        - ensure exactly 1 label1; if not, retry
        返回节点列表（原始 graph 的 node ids）或空列表失败
        """
        for attempt in range(max_attempts):
            # pick a center (prefer nodes with deg >= (size-1)/2)
            center = random.choice(list(self.G.nodes()))
            neighs = list(self.G.neighbors(center))
            if len(neighs) >= size - 1:
                choice = random.sample(neighs, size - 1)
                nodes = [center] + choice
            else:
                # 先取所有一跳，再补二跳
                nodes = [center]
                cur = neighs[:]
                random.shuffle(cur)
                nodes.extend(cur)
                if len(nodes) < size:
                    # add random neighbors-of-neighbors
                    added = []
                    for nb in neighs:
                        for nb2 in self.G.neighbors(nb):
                            if nb2 not in nodes:
                                added.append(nb2)
                    random.shuffle(added)
                    need = size - len(nodes)
                    nodes.extend(added[:need])
                # if still short, pick random nodes connected to center component
                if len(nodes) < size:
                    comp_nodes = list(nx.node_connected_component(self.G, center))
                    rem = [x for x in comp_nodes if x not in nodes]
                    if len(rem) >= (size - len(nodes)):
                        nodes.extend(random.sample(rem, size - len(nodes)))
                    else:
                        nodes.extend(rem)
                nodes = nodes[:size]

            if len(nodes) != size:
                continue
            if nx.is_connected(self.G.subgraph(nodes)) and self._ensure_exact_one_label1(nodes):
                return nodes
        return []

    def gen_path_one(self, size: int, max_attempts: int = 500) -> List[int]:
        """生成长度为 size 的简单路径（节点数 = size）"""
        for attempt in range(max_attempts):
            start = random.choice(list(self.G.nodes()))
            path = [start]
            visited = {start}
            while len(path) < size:
                cur = path[-1]
                nbrs = [v for v in self.G.neighbors(cur) if v not in visited]
                if not nbrs:
                    break
                nxt = random.choice(nbrs)
                path.append(nxt)
                visited.add(nxt)
            if len(path) < size:
                continue
            # ensure connected and exact one label1
            if self._ensure_exact_one_label1(path):
                return path
        return []

    def gen_cycle_one(self, size: int, max_attempts: int = 800) -> List[int]:
        """
        生成 size 节点的 cycle-like subgraph（近似）：随机找个连通子图并检查是否有一个 simple cycle 包含所有节点。
        更稳妥的方法：随机挑选一个起点，做随机游走直到节点数达到 size，然后检查是否首尾相连。
        """
        for attempt in range(max_attempts):
            start = random.choice(list(self.G.nodes()))
            walk = [start]
            visited = {start}
            while len(walk) < size:
                cur = walk[-1]
                nbrs = list(self.G.neighbors(cur))
                if not nbrs:
                    break
                nxt = random.choice(nbrs)
                # allow revisits but want unique nodes
                if nxt in visited:
                    # small chance to accept if it creates cycle late; else choose other
                    choices = [v for v in nbrs if v not in visited]
                    if choices:
                        nxt = random.choice(choices)
                    else:
                        # break to avoid infinite loop
                        break
                walk.append(nxt)
                visited.add(nxt)
            if len(walk) != size:
                continue
            # check if last connects back to first to make a cycle
            if self.G.has_edge(walk[-1], walk[0]):
                if self._ensure_exact_one_label1(walk):
                    return walk
            else:
                # try to connect endpoints via a short path inside the induced component
                sub = self.G.subgraph(walk)
                if nx.is_connected(sub):
                    # if there exists any edge between non-consecutive nodes to form a cycle covering all nodes
                    # as a fallback accept connected induced subgraph if it has at least size edges (dense)
                    edges = self._induced_subgraph_edges(walk)
                    if len(edges) >= size and self._ensure_exact_one_label1(walk):
                        return walk
        return []

    def gen_tree_one(self, size: int, max_attempts: int = 800) -> List[int]:
        """
        生成 size 节点的树状连通子图（BFS扩展保证无环优先）
        """
        for attempt in range(max_attempts):
            start = random.choice(list(self.G.nodes()))
            tree_nodes = {start}
            frontier = deque([start])
            while len(tree_nodes) < size and frontier:
                u = frontier.popleft()
                nbrs = [v for v in self.G.neighbors(u) if v not in tree_nodes]
                random.shuffle(nbrs)
                for v in nbrs[:3]:  # 每次扩展最多若干个分支
                    tree_nodes.add(v)
                    frontier.append(v)
                    if len(tree_nodes) >= size:
                        break
            if len(tree_nodes) < size:
                # try to expand with nodes from its connected component
                comp = list(nx.node_connected_component(self.G, start))
                rem = [x for x in comp if x not in tree_nodes]
                random.shuffle(rem)
                need = size - len(tree_nodes)
                tree_nodes.update(rem[:need])
            if len(tree_nodes) != size:
                continue
            sub = self.G.subgraph(tree_nodes)
            # if there are cycles, we can prune edges — but nodes set ok (we only need connected)
            if nx.is_connected(sub) and self._ensure_exact_one_label1(list(tree_nodes)):
                return list(tree_nodes)
        return []

    # ------------------- 批量生成控制 -------------------

    def generate_all(self, sizes: List[int], structures: List[str], N_per: int = 5, max_total_attempts_per_item: int = 1000):
        """
        对每个结构 & 每个 size 生成至少 N_per 个查询。
        structures 可以是 ["star","path","cycle","tree"]
        """
        generator_map = {
            "star": self.gen_star_one,
            "path": self.gen_path_one,
            "cycle": self.gen_cycle_one,
            "tree": self.gen_tree_one,
        }

        for struct in structures:
            if struct not in generator_map:
                print(f"[WARN] unknown structure {struct}, skipping.")
                continue
            gen_func = generator_map[struct]
            for size in sizes:
                out_subdir = os.path.join(self.out_dir, f"{struct}_{size}")
                os.makedirs(out_subdir, exist_ok=True)
                produced = 0
                tries = 0
                attempts_limit = max_total_attempts_per_item
                seen_node_sets = set()
                print(f"\n[INFO] Generating {N_per} queries for structure={struct}, size={size} ...")
                while produced < N_per and tries < attempts_limit:
                    tries += 1
                    nodes = gen_func(size)
                    if not nodes:
                        continue
                    nodes_sorted = tuple(sorted(nodes))
                    if nodes_sorted in seen_node_sets:
                        continue
                    seen_node_sets.add(nodes_sorted)
                    # write to file: ensure edges extracted from induced subgraph
                    edges = self._induced_subgraph_edges(list(nodes_sorted))
                    fname = os.path.join(out_subdir, f"query_{struct}_{size}_{produced}.graph")
                    self._write_fastest_query(list(nodes_sorted), edges, fname)
                    produced += 1
                if produced < N_per:
                    print(f"[WARN] Only produced {produced}/{N_per} for {struct} size {size} after {tries} tries.")
                else:
                    print(f"[OK] Produced {produced}/{N_per} queries for {struct} size {size} (tries={tries}).")


# ------------------- 运行示例 -------------------
if __name__ == "__main__":
    # 配置区
    dataset_name = 'dataset_four'
    data_graph_path = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/data_graph/parler.graph"  # 修改为实际路径
    out_dir = "/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/third_genegrate"
    sizes = [4, 6, 8]
    structures = ["star", "path", "cycle", "tree"]
    N = 100

    print("[INFO] Loading data graph ...")
    G = load_graph_fastest(data_graph_path)
    gen = QueryGenerator(G, out_dir=out_dir, seed=2025)
    gen.generate_all(sizes=sizes, structures=structures, N_per=N)
    print("[DONE] Generated queries under:", out_dir)


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
generate_queries.py

从带节点标签（0/1/2）、无边标签的 Fastest 格式图（.graph）自动生成查询 workload。
约束：
 - 结构类型： star, path, cycle, tree
 - 节点数： 4, 6, 8, 12
 - 每种结构 x 每种节点数 至少生成 N 个（可配置）
 - 每个查询中恰好包含 1 个 label==n 的节点（n 可配置，例如 Post=2）
 - 输出为 Fastest .graph（节点 0..n-1, 每行 v, 每行 e）
"""

import os
import random
from collections import deque
from typing import List
import networkx as nx

random.seed(42)


def load_graph_fastest(filepath: str) -> nx.Graph:
    """
    读取 Fastest .graph（或类似）格式的图。
    期望 v 行： v <vid> <label> [deg]
            e 行： e <u> <v> [label?]
    只读取节点标签和无向边。
    """
    G = nx.Graph()
    if not os.path.exists(filepath):
        raise FileNotFoundError(filepath)
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if parts[0] == "v" and len(parts) >= 3:
                vid = int(parts[1])
                label = int(parts[2])
                G.add_node(vid, label=label)
            elif parts[0] == "e" and len(parts) >= 3:
                u = int(parts[1])
                v = int(parts[2])
                G.add_edge(u, v)
    return G


class QueryGenerator:
    def __init__(self, G: nx.Graph, out_dir: str = "query_graphs", seed: int = 42, label_target: int = 2):
        self.G = G
        self.out_dir = out_dir
        self.label_target = label_target  # ✅ 指定必须包含的目标标签
        os.makedirs(self.out_dir, exist_ok=True)
        self.nodes_by_label = {}
        for n, d in self.G.nodes(data=True):
            lab = d.get("label", None)
            self.nodes_by_label.setdefault(lab, []).append(n)
        random.seed(seed)

    def _write_fastest_query(self, qnodes: List[int], edges: List[tuple], outpath: str):
        """将 query 写成 Fastest .graph：v idx label deg; e i j"""
        node_index = {n: i for i, n in enumerate(qnodes)}
        deg = {n: 0 for n in qnodes}
        for (u, v) in edges:
            deg[u] += 1
            deg[v] += 1

        with open(outpath, "w") as fout:
            fout.write(f"t {len(qnodes)} {len(edges)}\n")
            for n in qnodes:
                i = node_index[n]
                lab = int(self.G.nodes[n].get("label", 0))
                d = deg[n]
                fout.write(f"v {i} {lab} {d}\n")
            for (u, v) in edges:
                fout.write(f"e {node_index[u]} {node_index[v]}\n")

    def _induced_subgraph_edges(self, nodes: List[int]):
        """返回节点集的无向边（列表形式）"""
        edges = []
        s = set(nodes)
        for u in nodes:
            for v in self.G.neighbors(u):
                if v in s and u < v:
                    edges.append((u, v))
        return edges

    # ✅ 通用标签检查函数
    def _count_label(self, nodes: List[int], label_target: int) -> int:
        """统计节点集合中 label==label_target 的节点数量"""
        return sum(1 for n in nodes if int(self.G.nodes[n].get("label", 0)) == label_target)

    def _ensure_exact_one_label(self, nodes: List[int], label_target: int) -> bool:
        """检查节点集合是否恰好含 1 个 label==label_target"""
        return self._count_label(nodes, label_target) == 1

    # ------------------- 结构生成器 -------------------

    def gen_star_one(self, size: int, max_attempts: int = 500) -> List[int]:
        for attempt in range(max_attempts):
            center = random.choice(list(self.G.nodes()))
            neighs = list(self.G.neighbors(center))
            if len(neighs) >= size - 1:
                choice = random.sample(neighs, size - 1)
                nodes = [center] + choice
            else:
                nodes = [center]
                cur = neighs[:]
                random.shuffle(cur)
                nodes.extend(cur)
                if len(nodes) < size:
                    added = []
                    for nb in neighs:
                        for nb2 in self.G.neighbors(nb):
                            if nb2 not in nodes:
                                added.append(nb2)
                    random.shuffle(added)
                    need = size - len(nodes)
                    nodes.extend(added[:need])
                if len(nodes) < size:
                    comp_nodes = list(nx.node_connected_component(self.G, center))
                    rem = [x for x in comp_nodes if x not in nodes]
                    if len(rem) >= (size - len(nodes)):
                        nodes.extend(random.sample(rem, size - len(nodes)))
                    else:
                        nodes.extend(rem)
                nodes = nodes[:size]

            if len(nodes) != size:
                continue
            if nx.is_connected(self.G.subgraph(nodes)) and self._ensure_exact_one_label(nodes, self.label_target):
                return nodes
        return []

    def gen_path_one(self, size: int, max_attempts: int = 500) -> List[int]:
        for attempt in range(max_attempts):
            start = random.choice(list(self.G.nodes()))
            path = [start]
            visited = {start}
            while len(path) < size:
                cur = path[-1]
                nbrs = [v for v in self.G.neighbors(cur) if v not in visited]
                if not nbrs:
                    break
                nxt = random.choice(nbrs)
                path.append(nxt)
                visited.add(nxt)
            if len(path) < size:
                continue
            if self._ensure_exact_one_label(path, self.label_target):
                return path
        return []

    def gen_cycle_one(self, size: int, max_attempts: int = 800) -> List[int]:
        for attempt in range(max_attempts):
            start = random.choice(list(self.G.nodes()))
            walk = [start]
            visited = {start}
            while len(walk) < size:
                cur = walk[-1]
                nbrs = list(self.G.neighbors(cur))
                if not nbrs:
                    break
                nxt = random.choice(nbrs)
                if nxt in visited:
                    choices = [v for v in nbrs if v not in visited]
                    if choices:
                        nxt = random.choice(choices)
                    else:
                        break
                walk.append(nxt)
                visited.add(nxt)
            if len(walk) != size:
                continue
            if self.G.has_edge(walk[-1], walk[0]):
                if self._ensure_exact_one_label(walk, self.label_target):
                    return walk
            else:
                sub = self.G.subgraph(walk)
                if nx.is_connected(sub):
                    edges = self._induced_subgraph_edges(walk)
                    if len(edges) >= size and self._ensure_exact_one_label(walk, self.label_target):
                        return walk
        return []

    def gen_tree_one(self, size: int, max_attempts: int = 800) -> List[int]:
        for attempt in range(max_attempts):
            start = random.choice(list(self.G.nodes()))
            tree_nodes = {start}
            frontier = deque([start])
            while len(tree_nodes) < size and frontier:
                u = frontier.popleft()
                nbrs = [v for v in self.G.neighbors(u) if v not in tree_nodes]
                random.shuffle(nbrs)
                for v in nbrs[:3]:
                    tree_nodes.add(v)
                    frontier.append(v)
                    if len(tree_nodes) >= size:
                        break
            if len(tree_nodes) < size:
                comp = list(nx.node_connected_component(self.G, start))
                rem = [x for x in comp if x not in tree_nodes]
                random.shuffle(rem)
                need = size - len(tree_nodes)
                tree_nodes.update(rem[:need])
            if len(tree_nodes) != size:
                continue
            sub = self.G.subgraph(tree_nodes)
            if nx.is_connected(sub) and self._ensure_exact_one_label(list(tree_nodes), self.label_target):
                return list(tree_nodes)
        return []

    # ------------------- 批量生成控制 -------------------

    def generate_all(self, sizes: List[int], structures: List[str], N_per: int = 5, max_total_attempts_per_item: int = 1000):
        generator_map = {
            "star": self.gen_star_one,
            "path": self.gen_path_one,
            "cycle": self.gen_cycle_one,
            "tree": self.gen_tree_one,
        }

        for struct in structures:
            if struct not in generator_map:
                print(f"[WARN] unknown structure {struct}, skipping.")
                continue
            gen_func = generator_map[struct]
            for size in sizes:
                out_subdir = os.path.join(self.out_dir, f"{struct}_{size}")
                os.makedirs(out_subdir, exist_ok=True)
                produced = 0
                tries = 0
                attempts_limit = max_total_attempts_per_item
                seen_node_sets = set()
                print(f"\n[INFO] Generating {N_per} queries for structure={struct}, size={size} ...")
                while produced < N_per and tries < attempts_limit:
                    tries += 1
                    nodes = gen_func(size)
                    if not nodes:
                        continue
                    nodes_sorted = tuple(sorted(nodes))
                    if nodes_sorted in seen_node_sets:
                        continue
                    seen_node_sets.add(nodes_sorted)
                    edges = self._induced_subgraph_edges(list(nodes_sorted))
                    fname = os.path.join(out_subdir, f"query_{struct}_{size}_{produced}.graph")
                    self._write_fastest_query(list(nodes_sorted), edges, fname)
                    produced += 1
                if produced < N_per:
                    print(f"[WARN] Only produced {produced}/{N_per} for {struct} size {size} after {tries} tries.")
                else:
                    print(f"[OK] Produced {produced}/{N_per} queries for {struct} size {size} (tries={tries}).")


# ------------------- 运行示例 -------------------
if __name__ == "__main__":
    dataset_name = 'dataset_four'
    data_graph_path = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/data_graph/parler.graph"
    out_dir = "/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/four_generate"
    sizes = [4, 6,8]
    structures = ["star", "path", "cycle", "tree"]
    N = 50
    label_target = 2  # ✅ 每个查询必须包含恰好一个 label==2 的节点

    print("[INFO] Loading data graph ...")
    G = load_graph_fastest(data_graph_path)
    gen = QueryGenerator(G, out_dir=out_dir, seed=2025, label_target=label_target)
    gen.generate_all(sizes=sizes, structures=structures, N_per=N)
    print("[DONE] Generated queries under:", out_dir)


#### 1.2 精确划分1.1生成的查询图结构：（因为1.1无法生成精确的查询图结构） 
已经运行完了生成查询图的代码，将生成的各种随机查询图保存到了/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/third_genegrate 文件夹下，文件夹下的内容分为，
cycle_4  cycle_6  cycle_8  path_4  path_6  path_8  star_4  star_6  star_8  tree_4  tree_6  tree_8
虽然是按照图结构和节点数量划分的文件夹，但是实际生成的查询图并不是严格满足该结构，我想依次读取每个文件夹下的查询图，写判定算法再次判断是否是path\star\tree\cycle结构，然后重新划分到/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/third_genegrate/precision_structure 文件夹下path、star、tree、cycle四个文件夹中

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import shutil
from collections import defaultdict, deque

# === 路径设置 ===
gen_data_name ='controlled_generation_v2'
base_dir = f"/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/{gen_data_name}"
output_dir = os.path.join(base_dir, "precision_structure")

# === 创建输出文件夹 ===
for name in ["path", "star", "tree", "cycle"]:
    os.makedirs(os.path.join(output_dir, name), exist_ok=True)


def read_graph(filepath):
    """读取 .graph 文件，返回节点列表和边列表"""
    edges = []
    nodes = set()
    with open(filepath, "r") as f:
        for line in f:
            if line.startswith("v "):
                parts = line.strip().split()
                nodes.add(int(parts[1]))
            elif line.startswith("e "):
                parts = line.strip().split()
                u, v = int(parts[1]), int(parts[2])
                edges.append((u, v))
                nodes.update([u, v])
    return list(nodes), edges


def build_adj(nodes, edges):
    adj = defaultdict(list)
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    return adj


def is_connected(nodes, adj):
    """判断图是否连通"""
    if not nodes:
        return False
    visited = set()
    q = deque([nodes[0]])
    while q:
        u = q.popleft()
        if u in visited:
            continue
        visited.add(u)
        for v in adj[u]:
            if v not in visited:
                q.append(v)
    return len(visited) == len(nodes)


def has_cycle(nodes, adj):
    """DFS 判断是否有环"""
    visited = set()

    def dfs(u, parent):
        visited.add(u)
        for v in adj[u]:
            if v not in visited:
                if dfs(v, u):
                    return True
            elif v != parent:
                return True
        return False

    for node in nodes:
        if node not in visited:
            if dfs(node, -1):
                return True
    return False


def classify_graph(nodes, edges):
    """严格分类为 path, star, tree, cycle"""
    adj = build_adj(nodes, edges)
    degrees = [len(adj[n]) for n in nodes]
    edge_count = len(edges)
    node_count = len(nodes)

    # 若不连通，忽略
    if not is_connected(nodes, adj):
        return None

    # === 有环 ===
    if has_cycle(nodes, adj):
        return "cycle"

    # === Path ===
    if degrees.count(1) == 2 and all(d in (1, 2) for d in degrees):
        return "path"

    # === Star ===
    centers = [d for d in degrees if d > 2]
    if len(centers) == 1 and degrees.count(1) == node_count - 1:
        return "star"

    # === Tree ===
    if edge_count == node_count - 1:
        return "tree"

    return None


# === 主流程 ===
for subdir in sorted(os.listdir(base_dir)):
    sub_path = os.path.join(base_dir, subdir)
    if not os.path.isdir(sub_path):
        continue

    for fname in os.listdir(sub_path):
        if not fname.endswith(".graph"):
            continue

        fpath = os.path.join(sub_path, fname)
        nodes, edges = read_graph(fpath)
        gtype = classify_graph(nodes, edges)

        if not gtype:
            continue  # 非连通或无效图跳过

        dst = os.path.join(output_dir, gtype, fname)
        shutil.copy2(fpath, dst)

        print(f"[{gtype.upper():7}] Copied: {fname}")

print("✅ 精确结构分类完成（有环全部归为 cycle）！")


1.3 精确分类后重新命名查询图并去重：
已经精确分类好cycle\star\path\tree的查询，重新为这些分好类的查询编码和命名query_[结构类型]_[顶点数量]_[第几个（从零开始编码）]

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
import tempfile
import shutil
import os
import hashlib

gen_data_name ='controlled_generation_v2'
base_dir = f"/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/{gen_data_name}/precision_structure"
structures = ["cycle", "star", "path", "tree"]

def file_hash(path):
    """计算文件内容哈希（忽略首尾空格、换行差异）"""
    with open(path, "r") as f:
        content = "".join(line.strip() for line in f if line.strip())
    return hashlib.md5(content.encode("utf-8")).hexdigest()


def deduplicate_graphs(struct_dir):
    """去重：删除内容相同的文件"""
    if not os.path.isdir(struct_dir):
        print(f"[WARN] missing dir {struct_dir}")
        return 0

    files = sorted([f for f in os.listdir(struct_dir) if f.endswith(".graph")])
    seen_hashes = set()
    duplicate_count = 0

    for f in files:
        fpath = os.path.join(struct_dir, f)
        h = file_hash(fpath)
        if h in seen_hashes:
            print(f"[DUPLICATE] removed -> {f}")
            os.remove(fpath)
            duplicate_count += 1
        else:
            seen_hashes.add(h)

    return duplicate_count


def rename_graphs_safe(struct_dir, struct):
    """安全重命名：先放临时目录，避免覆盖冲突"""
    if not os.path.isdir(struct_dir):
        return

    files = sorted([f for f in os.listdir(struct_dir) if f.endswith(".graph")])
    counter = 0

    with tempfile.TemporaryDirectory() as tmpdir:
        for f in files:
            fpath = os.path.join(struct_dir, f)

            # 统计顶点数量
            node_count = 0
            with open(fpath, "r") as infile:
                for line in infile:
                    if line.startswith("t "):
                        parts = line.strip().split()
                        if len(parts) >= 2:
                            node_count = int(parts[1])
                            break
                    elif line.startswith("v "):
                        node_count += 1

            # 生成新文件名
            new_name = f"query_{struct}_{node_count}_{counter}.graph"
            tmp_path = os.path.join(tmpdir, new_name)
            shutil.copy(fpath, tmp_path)
            counter += 1

        # 清空原目录再复制回去
        for f in files:
            os.remove(os.path.join(struct_dir, f))
        for f in os.listdir(tmpdir):
            shutil.move(os.path.join(tmpdir, f), os.path.join(struct_dir, f))
            print(f"[RENAME] -> {f}")

# ------------------ 主程序 ------------------
for struct in structures:
    struct_dir = os.path.join(base_dir, struct)
    
    duplicates = deduplicate_graphs(struct_dir)
    print(f"[INFO] {struct.upper()} 去重完成，共删除 {duplicates} 个重复文件")
    
    rename_graphs_safe(struct_dir, struct)

print("✅ 所有结构去重并重新命名完成！")


#### 1.4 cycle to tree
经过上面操作后，发现cycle的查询图数量最多，我想基于cycle的查询图通过尝试删除一些边，变成tree类型的查询图，要求删除边后查询图仍然是连通的，并且不能和已有的tree查询图重复，帮我写代码实现这个功能，目标是将cycle查询图转换成tree查询图，直到tree查询图数量达到cycle查询图数量的一半为止或者没有更多cycle查询图可以转换为止

我的数据图我顶点有标签（0 代表user标签 ，1代表post标签，2代表comment标签），但是边没有了标签（只是将上面五种边的标签去掉，且将边标签为0和4的边合并了） 修改下面代码，生成的tree查询图依然要满足标签为1的点有一个，我的图结构的意义如上 ，而且comment只能连接多个comment，只能连接一个user和一个post。

生成不是路径和星查询的经典树结构的树查询图

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import hashlib
from collections import defaultdict, deque

base_dir = f"/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/{gen_data_name}/precision_structure"
cycle_dir = os.path.join(base_dir, "cycle")
tree_dir = os.path.join(base_dir, "tree")

os.makedirs(tree_dir, exist_ok=True)

# ------------------ 辅助函数 ------------------

def canonical_representation(node_labels, edges):
    nodes = sorted(node_labels.keys())
    mapping = {old: new for new, old in enumerate(nodes)}
    deg = {old: 0 for old in nodes}
    edge_set = set()
    for u, v in edges:
        if u not in node_labels or v not in node_labels:
            continue
        a, b = min(u, v), max(u, v)
        edge_set.add((a, b))
        deg[a] += 1
        deg[b] += 1

    n = len(nodes)
    m = len(edge_set)
    lines = []
    lines.append(f"t {n} {m}")
    for old in nodes:
        i = mapping[old]
        lbl = node_labels[old]
        d = deg[old]
        lines.append(f"v {i} {lbl} {d}")
    edges_new = sorted([(mapping[a], mapping[b]) for (a, b) in edge_set])
    for (i, j) in edges_new:
        lines.append(f"e {i} {j}")
    rep = "\n".join(lines) + "\n"
    return rep, mapping

def content_hash_from_rep(rep):
    return hashlib.md5(rep.encode("utf-8")).hexdigest()

def parse_graph(path):
    node_labels = {}
    edges = []
    with open(path, "r") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            if s.startswith("v "):
                parts = s.split()
                if len(parts) >= 3:
                    nid = int(parts[1])
                    lbl = int(parts[2])
                    node_labels[nid] = lbl
            elif s.startswith("e "):
                parts = line.split()
                if len(parts) >= 3:
                    u, v = int(parts[1]), int(parts[2])
                    edges.append((u, v))
    return node_labels, edges

def is_connected(nodes, edges):
    if not nodes:
        return False
    adj = defaultdict(list)
    for u, v in edges:
        if u in nodes and v in nodes:
            adj[u].append(v)
            adj[v].append(u)
    start = next(iter(nodes))
    visited = set()
    q = deque([start])
    while q:
        x = q.popleft()
        if x in visited:
            continue
        visited.add(x)
        for nei in adj[x]:
            if nei not in visited:
                q.append(nei)
    return len(visited) == len(nodes)

def has_cycle(nodes, edges):
    adj = defaultdict(list)
    for u, v in edges:
        if u in nodes and v in nodes:
            adj[u].append(v)
            adj[v].append(u)

    visited = set()
    def dfs(u, parent):
        visited.add(u)
        for v in adj[u]:
            if v == parent:
                continue
            if v in visited or dfs(v, u):
                return True
        return False

    for node in nodes:
        if node not in visited:
            if dfs(node, -1):
                return True
    return False

def check_semantic_constraints(node_labels, edges):
    post_nodes = [n for n, lbl in node_labels.items() if lbl == 1]
    if len(post_nodes) != 1:
        return False
    adj = defaultdict(list)
    valid_pairs = {(0,1),(1,0),(0,2),(2,0),(1,2),(2,1),(2,2)}
    for u,v in edges:
        if u not in node_labels or v not in node_labels:
            return False
        lu, lv = node_labels[u], node_labels[v]
        if (lu, lv) not in valid_pairs:
            return False
        adj[u].append(v)
        adj[v].append(u)
    for n,lbl in node_labels.items():
        if lbl == 2:
            user_count = sum(1 for nei in adj[n] if node_labels[nei]==0)
            post_count = sum(1 for nei in adj[n] if node_labels[nei]==1)
            if user_count>1 or post_count>1:
                return False
    return True

def load_existing_tree_hashes(tree_dir):
    hashes = set()
    for fname in os.listdir(tree_dir):
        if not fname.endswith(".graph"):
            continue
        path = os.path.join(tree_dir, fname)
        node_labels, edges = parse_graph(path)
        rep, _ = canonical_representation(node_labels, edges)
        hashes.add(content_hash_from_rep(rep))
    return hashes

def is_path_tree(node_labels, edges):
    deg = defaultdict(int)
    for u,v in edges:
        deg[u] += 1
        deg[v] += 1
    return max(deg.values(), default=0) <= 2

def is_star_tree(node_labels, edges):
    deg = defaultdict(int)
    for u,v in edges:
        deg[u] += 1
        deg[v] += 1
    # 星形特点：一个节点度==n-1，其余度==1
    n = len(node_labels)
    deg_values = list(deg.values())
    if n<=2:
        return False
    return deg_values.count(1) == n-1 and (n-1) in deg_values

# ------------------ 主程序 ------------------

cycle_files = sorted([f for f in os.listdir(cycle_dir) if f.endswith(".graph")])
existing_tree_hashes = load_existing_tree_hashes(tree_dir)
tree_files = [f for f in os.listdir(tree_dir) if f.endswith(".graph")]

target_tree_count = len(cycle_files)//2
current_tree_count = len(tree_files)
counter = len(tree_files)

print(f"[INFO] 当前 tree 数量: {current_tree_count}, 目标: {target_tree_count}")

for f in cycle_files:
    if current_tree_count >= target_tree_count:
        break

    fpath = os.path.join(cycle_dir, f)
    node_labels, edges = parse_graph(fpath)
    nodes = set(node_labels.keys())

    if len(node_labels)==0 or len(edges)<=len(nodes)-1:
        continue

    converted = False
    for i,(u,v) in enumerate(edges):
        new_edges = edges[:i]+edges[i+1:]

        if not is_connected(nodes, new_edges):
            continue
        if has_cycle(nodes,new_edges):
            continue
        if not check_semantic_constraints(node_labels,new_edges):
            continue
        if is_path_tree(node_labels,new_edges) or is_star_tree(node_labels,new_edges):
            continue

        rep,_ = canonical_representation(node_labels,new_edges)
        h = content_hash_from_rep(rep)
        if h in existing_tree_hashes:
            continue

        new_name = f"query_tree_{len(nodes)}_{counter}.graph"
        out_path = os.path.join(tree_dir,new_name)
        with open(out_path,"w") as fo:
            fo.write(rep)
        existing_tree_hashes.add(h)
        counter +=1
        current_tree_count +=1
        converted = True
        print(f"[NEW TREE] {new_name} <- from cycle {f} (deleted edge {u}-{v})")
        break

print(f"✅ 完成：新增 {current_tree_count-len(tree_files)} 个 tree（当前 tree 数量 {current_tree_count}）")


#### 1.5 cycle to star
经过上面操作后，发现cycle的查询图数量最多，我想基于cycle的查询图通过尝试删除一些边，变成星形的查询图，要求删除边后查询图仍然是连通的，并且不能和已有的星形查询图重复，帮我写代码实现这个功能，目标是将cycle查询图转换成星形查询图，直到星形查询图数量达到cycle查询图数量的一半为止或者没有更多cycle查询图可以转换为止，星形为经典的星形，不是路径也不是树


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import hashlib
from collections import defaultdict, deque
from itertools import combinations

base_dir = "/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/four_generate/precision_structure"
cycle_dir = os.path.join(base_dir, "cycle")
star_dir = os.path.join(base_dir, "star")
os.makedirs(star_dir, exist_ok=True)

# ------------------ 辅助函数 ------------------

def canonical_representation(node_labels, edges):
    nodes = sorted(node_labels.keys())
    mapping = {old: new for new, old in enumerate(nodes)}
    deg = {old: 0 for old in nodes}
    edge_set = set()
    for u, v in edges:
        if u not in node_labels or v not in node_labels:
            continue
        a, b = min(u, v), max(u, v)
        edge_set.add((a, b))
        deg[a] += 1
        deg[b] += 1

    n = len(nodes)
    m = len(edge_set)
    lines = []
    lines.append(f"t {n} {m}")
    for old in nodes:
        i = mapping[old]
        lbl = node_labels[old]
        d = deg[old]
        lines.append(f"v {i} {lbl} {d}")
    edges_new = sorted([(mapping[a], mapping[b]) for (a, b) in edge_set])
    for i,j in edges_new:
        lines.append(f"e {i} {j}")
    rep = "\n".join(lines) + "\n"
    return rep, mapping

def content_hash_from_rep(rep):
    return hashlib.md5(rep.encode("utf-8")).hexdigest()

def parse_graph(path):
    node_labels = {}
    edges = []
    with open(path, "r") as f:
        for line in f:
            s = line.strip()
            if not s: continue
            if s.startswith("v "):
                parts = s.split()
                if len(parts)>=3:
                    nid = int(parts[1])
                    lbl = int(parts[2])
                    node_labels[nid] = lbl
            elif s.startswith("e "):
                parts = line.split()
                if len(parts)>=3:
                    u, v = int(parts[1]), int(parts[2])
                    edges.append((u, v))
    return node_labels, edges

def is_connected(nodes, edges):
    if not nodes: return False
    adj = defaultdict(list)
    for u,v in edges:
        if u in nodes and v in nodes:
            adj[u].append(v)
            adj[v].append(u)
    start = next(iter(nodes))
    visited = set()
    q = deque([start])
    while q:
        x = q.popleft()
        if x in visited: continue
        visited.add(x)
        for nei in adj[x]:
            if nei not in visited: q.append(nei)
    return len(visited) == len(nodes)

def has_cycle(nodes, edges):
    adj = defaultdict(list)
    for u,v in edges:
        if u in nodes and v in nodes:
            adj[u].append(v)
            adj[v].append(u)
    visited = set()
    def dfs(u,parent):
        visited.add(u)
        for v in adj[u]:
            if v==parent: continue
            if v in visited or dfs(v,u): return True
        return False
    for node in nodes:
        if node not in visited:
            if dfs(node,-1): return True
    return False

def check_semantic_constraints(node_labels, edges):
    post_nodes = [n for n,lbl in node_labels.items() if lbl==1]
    if len(post_nodes)!=1: return False
    adj = defaultdict(list)
    valid_pairs = {(0,1),(1,0),(0,2),(2,0),(1,2),(2,1),(2,2)}
    for u,v in edges:
        if u not in node_labels or v not in node_labels: return False
        lu, lv = node_labels[u], node_labels[v]
        if (lu,lv) not in valid_pairs: return False
        adj[u].append(v)
        adj[v].append(u)
    for n,lbl in node_labels.items():
        if lbl==2:
            user_count = sum(1 for nei in adj[n] if node_labels[nei]==0)
            post_count = sum(1 for nei in adj[n] if node_labels[nei]==1)
            if user_count>1 or post_count>1: return False
    return True

def load_existing_star_hashes(star_dir):
    hashes = set()
    for fname in os.listdir(star_dir):
        if not fname.endswith(".graph"): continue
        path = os.path.join(star_dir,fname)
        node_labels, edges = parse_graph(path)
        rep,_ = canonical_representation(node_labels, edges)
        hashes.add(content_hash_from_rep(rep))
    return hashes

def is_classic_star_tree(nodes, edges):
    deg = defaultdict(int)
    for u,v in edges:
        deg[u]+=1
        deg[v]+=1
    n = len(nodes)
    if n<=2: return False
    deg_values = list(deg.values())
    return deg_values.count(1) == n-1 and (n-1) in deg_values

# ------------------ 主程序 ------------------

cycle_files = sorted([f for f in os.listdir(cycle_dir) if f.endswith(".graph")])
existing_star_hashes = load_existing_star_hashes(star_dir)
star_files = [f for f in os.listdir(star_dir) if f.endswith(".graph")]

target_star_count = len(cycle_files)//2
current_star_count = len(star_files)
counter = len(star_files)

print(f"[INFO] 当前 star 数量: {current_star_count}, 目标: {target_star_count}")

for f in cycle_files:
    if current_star_count >= target_star_count: break
    fpath = os.path.join(cycle_dir,f)
    node_labels, edges = parse_graph(fpath)
    nodes = set(node_labels.keys())
    n_nodes = len(nodes)
    if len(node_labels)==0 or len(edges)<n_nodes-1: continue

    # 尝试以每个节点作为中心生成星形
    for center in nodes:
        star_edges = [(center, other) for other in nodes if other!=center]
        if not is_connected(nodes, star_edges): continue
        if has_cycle(nodes, star_edges): continue
        if not check_semantic_constraints(node_labels, star_edges): continue
        if not is_classic_star_tree(nodes, star_edges): continue

        rep,_ = canonical_representation(node_labels, star_edges)
        h = content_hash_from_rep(rep)
        if h in existing_star_hashes: continue

        new_name = f"query_star_{n_nodes}_{counter}.graph"
        out_path = os.path.join(star_dir,new_name)
        with open(out_path,"w") as fo:
            fo.write(rep)
        existing_star_hashes.add(h)
        counter+=1
        current_star_count+=1
        print(f"[NEW STAR] {new_name} <- from cycle {f} (center {center})")
        break

print(f"✅ 完成：新增 {current_star_count-len(star_files)} 个 star（当前 star 数量 {current_star_count}）")


#### 1.6 cycle to path
经过上面操作后，发现cycle的查询图数量最多，我想基于cycle的查询图通过尝试删除一些边，变成路径的查询图，要求删除边后查询图仍然是连通的，并且不能和已有的路径查询图重复，帮我写代码实现这个功能，目标是将cycle查询图转换成星形查询图，直到路径查询图数量达到cycle查询图数量的一半为止或者没有更多cycle查询图可以转换为止，路径就是一条直线，不是树也不是星更不是cycle

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import hashlib
from collections import defaultdict, deque

base_dir = "/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/four_generate/precision_structure"
cycle_dir = os.path.join(base_dir, "cycle")
path_dir = os.path.join(base_dir, "path")
os.makedirs(path_dir, exist_ok=True)

# ------------------ 辅助函数 ------------------

def canonical_representation(node_labels, edges):
    nodes = sorted(node_labels.keys())
    mapping = {old: new for new, old in enumerate(nodes)}
    deg = {old: 0 for old in nodes}
    edge_set = set()
    for u, v in edges:
        if u not in node_labels or v not in node_labels:
            continue
        a, b = min(u, v), max(u, v)
        edge_set.add((a, b))
        deg[a] += 1
        deg[b] += 1

    n = len(nodes)
    m = len(edge_set)
    lines = []
    lines.append(f"t {n} {m}")
    for old in nodes:
        i = mapping[old]
        lbl = node_labels[old]
        d = deg[old]
        lines.append(f"v {i} {lbl} {d}")
    edges_new = sorted([(mapping[a], mapping[b]) for (a, b) in edge_set])
    for i,j in edges_new:
        lines.append(f"e {i} {j}")
    rep = "\n".join(lines) + "\n"
    return rep, mapping

def content_hash_from_rep(rep):
    return hashlib.md5(rep.encode("utf-8")).hexdigest()

def parse_graph(path):
    node_labels = {}
    edges = []
    with open(path, "r") as f:
        for line in f:
            s = line.strip()
            if not s: continue
            if s.startswith("v "):
                parts = s.split()
                if len(parts)>=3:
                    nid = int(parts[1])
                    lbl = int(parts[2])
                    node_labels[nid] = lbl
            elif s.startswith("e "):
                parts = line.split()
                if len(parts)>=3:
                    u, v = int(parts[1]), int(parts[2])
                    edges.append((u, v))
    return node_labels, edges

def is_connected(nodes, edges):
    if not nodes: return False
    adj = defaultdict(list)
    for u,v in edges:
        if u in nodes and v in nodes:
            adj[u].append(v)
            adj[v].append(u)
    start = next(iter(nodes))
    visited = set()
    q = deque([start])
    while q:
        x = q.popleft()
        if x in visited: continue
        visited.add(x)
        for nei in adj[x]:
            if nei not in visited: q.append(nei)
    return len(visited) == len(nodes)

def has_cycle(nodes, edges):
    adj = defaultdict(list)
    for u,v in edges:
        if u in nodes and v in nodes:
            adj[u].append(v)
            adj[v].append(u)
    visited = set()
    def dfs(u,parent):
        visited.add(u)
        for v in adj[u]:
            if v==parent: continue
            if v in visited or dfs(v,u): return True
        return False
    for node in nodes:
        if node not in visited:
            if dfs(node,-1): return True
    return False

def check_semantic_constraints(node_labels, edges):
    post_nodes = [n for n,lbl in node_labels.items() if lbl==1]
    if len(post_nodes)!=1: return False
    adj = defaultdict(list)
    valid_pairs = {(0,1),(1,0),(0,2),(2,0),(1,2),(2,1),(2,2)}
    for u,v in edges:
        if u not in node_labels or v not in node_labels: return False
        lu, lv = node_labels[u], node_labels[v]
        if (lu,lv) not in valid_pairs: return False
        adj[u].append(v)
        adj[v].append(u)
    for n,lbl in node_labels.items():
        if lbl==2:
            user_count = sum(1 for nei in adj[n] if node_labels[nei]==0)
            post_count = sum(1 for nei in adj[n] if node_labels[nei]==1)
            if user_count>1 or post_count>1: return False
    return True

def load_existing_path_hashes(path_dir):
    hashes = set()
    for fname in os.listdir(path_dir):
        if not fname.endswith(".graph"): continue
        path = os.path.join(path_dir,fname)
        node_labels, edges = parse_graph(path)
        rep,_ = canonical_representation(node_labels, edges)
        hashes.add(content_hash_from_rep(rep))
    return hashes

def is_path_tree(nodes, edges):
    # Path: 两个节点度=1，其余度=2
    deg = defaultdict(int)
    for u,v in edges:
        deg[u]+=1
        deg[v]+=1
    if len(edges) != len(nodes)-1: return False
    deg_values = list(deg.values())
    ones = deg_values.count(1)
    twos = deg_values.count(2)
    return ones==2 and twos==len(nodes)-2

# ------------------ 主程序 ------------------

cycle_files = sorted([f for f in os.listdir(cycle_dir) if f.endswith(".graph")])
existing_path_hashes = load_existing_path_hashes(path_dir)
path_files = [f for f in os.listdir(path_dir) if f.endswith(".graph")]

target_path_count = len(cycle_files)//2
current_path_count = len(path_files)
counter = len(path_files)

print(f"[INFO] 当前 path 数量: {current_path_count}, 目标: {target_path_count}")

for f in cycle_files:
    if current_path_count >= target_path_count: break
    fpath = os.path.join(cycle_dir,f)
    node_labels, edges = parse_graph(fpath)
    nodes = set(node_labels.keys())
    n_nodes = len(nodes)
    if len(node_labels)==0 or len(edges)<n_nodes-1: continue

    # 尝试删除多条边使 graph 变为 path
    # 简单策略：所有 edge 子集长度=n_nodes-1，检查连通、无环、语义、是否 path
    from itertools import combinations
    for subset in combinations(edges, n_nodes-1):
        subset_edges = list(subset)
        if not is_connected(nodes, subset_edges): continue
        if has_cycle(nodes, subset_edges): continue
        if not check_semantic_constraints(node_labels, subset_edges): continue
        if not is_path_tree(nodes, subset_edges): continue

        rep,_ = canonical_representation(node_labels, subset_edges)
        h = content_hash_from_rep(rep)
        if h in existing_path_hashes: continue

        new_name = f"query_path_{n_nodes}_{counter}.graph"
        out_path = os.path.join(path_dir,new_name)
        with open(out_path,"w") as fo:
            fo.write(rep)
        existing_path_hashes.add(h)
        counter+=1
        current_path_count+=1
        print(f"[NEW PATH] {new_name} <- from cycle {f}")
        break  # 每个 cycle 只生成一个 path

print(f"✅ 完成：新增 {current_path_count-len(path_files)} 个 path（当前 path 数量 {current_path_count}）")


#### 1.7 统计重复查询图数量：
 /home/wangshuo/resource/datasets/parler_data/proto_query_graphs/third_genegrate/precision_structure 上面文件夹下有四个文件夹cycle\path\star\tree，读取这四个文件夹下所有的查询图，统计相同查询图的个数，并输出出来

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import hashlib
from collections import defaultdict

base_dir = "/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/four_generate/precision_structure"
structures = ["cycle", "star", "path", "tree"]

def parse_graph(path):
    """解析 .graph 文件，返回 node_labels (dict) 和 edges (list of tuples)"""
    node_labels = {}
    edges = []
    with open(path, "r") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            if s.startswith("v "):
                parts = s.split()
                if len(parts) >= 3:
                    nid = int(parts[1])
                    lbl = int(parts[2])
                    node_labels[nid] = lbl
            elif line.startswith("e "):
                parts = line.split()
                if len(parts) >= 3:
                    u, v = int(parts[1]), int(parts[2])
                    edges.append((u, v))
    return node_labels, edges


def canonical_hash(node_labels, edges):
    """生成图结构哈希（忽略节点编号顺序和行顺序）"""
    nodes = sorted(node_labels.keys())
    mapping = {old: new for new, old in enumerate(nodes)}
    
    # 重新编号边
    edges_new = sorted([(min(mapping[u], mapping[v]), max(mapping[u], mapping[v])) for u, v in edges])
    
    # 节点标签列表
    labels_sorted = [node_labels[n] for n in nodes]
    
    # 构造唯一字符串
    rep_str = ",".join(map(str, labels_sorted)) + "|" + "|".join(f"{u}-{v}" for u, v in edges_new)
    
    return hashlib.md5(rep_str.encode("utf-8")).hexdigest()


for struct in structures:
    struct_dir = os.path.join(base_dir, struct)
    if not os.path.isdir(struct_dir):
        print(f"[WARN] missing dir {struct_dir}")
        continue

    files = [f for f in os.listdir(struct_dir) if f.endswith(".graph")]
    hash_count = defaultdict(int)
    
    for f in files:
        fpath = os.path.join(struct_dir, f)
        node_labels, edges = parse_graph(fpath)
        h = canonical_hash(node_labels, edges)
        hash_count[h] += 1
    
    duplicates = sum(count - 1 for count in hash_count.values() if count > 1)
    print(f"[INFO] {struct.upper():6} 共有 {len(files)} 个文件，其中重复查询图 {duplicates} 个")


#### 1.8 检验查询图合法性并删除不合法的查询图：
/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/third_genegrate/precision_structure 上面文件夹下有四个文件夹cycle\path\star\tree，读取这四个文件夹下所有的查询图，看看有多少满足以下要求的查询：
1.连通
2.每个用户可以连接多个post，多个comment，post可以连接多个用户，多个comment，但一个comment可以连接多个comment，只能连接一个用户和一个post
3.label =2 的查询点每个查询只有一个
4.同时还要保证生成查询图的节点第一列是v，第二列是id（从0开始），第三列是标签，第四列是度数

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
from collections import defaultdict, deque

base_dir = "/home/wangshuo/resource/datasets/parler_data/proto_query_graphs/four_generate/precision_structure"
structures = ["cycle", "star", "path", "tree"]

# ------------------ 辅助函数 ------------------

def parse_graph(path):
    """解析 .graph 文件，返回 node_labels (dict) 和 edges (list of tuples)"""
    node_labels = {}
    edges = []
    with open(path, "r") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            if s.startswith("v "):
                parts = s.split()
                if len(parts) >= 3:
                    nid = int(parts[1])
                    lbl = int(parts[2])
                    node_labels[nid] = lbl
            elif s.startswith("e "):
                parts = line.split()
                if len(parts) >= 3:
                    u, v = int(parts[1]), int(parts[2])
                    edges.append((u, v))
    return node_labels, edges


def is_connected(nodes, edges):
    """判断图是否连通"""
    if not nodes:
        return False
    adj = defaultdict(list)
    for u, v in edges:
        if u in nodes and v in nodes:
            adj[u].append(v)
            adj[v].append(u)
    start = next(iter(nodes))
    visited = set()
    q = deque([start])
    while q:
        x = q.popleft()
        if x in visited:
            continue
        visited.add(x)
        for nei in adj[x]:
            if nei not in visited:
                q.append(nei)
    return len(visited) == len(nodes)


def check_constraints(node_labels, edges):
    """检查语义约束，返回 (是否满足, 不满足原因)"""
    # 唯一 post (label==1)
    post_nodes = [n for n, lbl in node_labels.items() if lbl == 2]
    if len(post_nodes) != 1:
        return False, f"不唯一的 post 节点，数量={len(post_nodes)}"

    # 构建邻接
    adj = defaultdict(list)
    for u, v in edges:
        if u not in node_labels or v not in node_labels:
            return False, f"存在边连接不存在的节点 {u}-{v}"
        lu, lv = node_labels[u], node_labels[v]

        # 检查边类型合法性
        valid_pairs = {(0, 1), (1, 0), (0, 2), (2, 0), (1, 2), (2, 1), (2, 2)}
        if (lu, lv) not in valid_pairs:
            return False, f"非法边类型: {lu}-{lv}"

        adj[u].append(v)
        adj[v].append(u)

    # comment 的连接限制：≤1 user, ≤1 post, 可以多个 comment
    for n, lbl in node_labels.items():
        if lbl == 2:
            user_count = sum(1 for nei in adj[n] if node_labels[nei] == 0)
            post_count = sum(1 for nei in adj[n] if node_labels[nei] == 1)
            if user_count > 1 or post_count > 1:
                return False, f"comment 节点 {n} 连接超过限制 (user={user_count}, post={post_count})"

    return True, ""


def canonicalize_graph(node_labels, edges):
    """生成规范化 graph 字符串：v id label degree"""
    nodes = sorted(node_labels.keys())
    mapping = {old: new for new, old in enumerate(nodes)}  # old id -> 0..n-1
    deg = {old: 0 for old in nodes}
    edge_set = set()
    for u, v in edges:
        if u not in node_labels or v not in node_labels:
            continue
        a, b = min(u, v), max(u, v)
        edge_set.add((a, b))
        deg[a] += 1
        deg[b] += 1

    n = len(nodes)
    m = len(edge_set)
    lines = [f"t {n} {m}"]
    for old in nodes:
        i = mapping[old]
        lbl = node_labels[old]
        d = deg[old]
        lines.append(f"v {i} {lbl} {d}")

    edges_new = sorted([(mapping[a], mapping[b]) for (a, b) in edge_set])
    for i, j in edges_new:
        lines.append(f"e {i} {j}")

    return "\n".join(lines) + "\n"


def check_node_ids(node_labels):
    """检查节点 id 是否从 0 开始且连续"""
    if not node_labels:
        return False, "没有节点"
    ids = sorted(node_labels.keys())
    expected = list(range(len(ids)))
    if ids != expected:
        return False, f"节点 id 不连续或不从0开始: {ids}"
    return True, ""


# ------------------ 主程序 ------------------

for struct in structures:
    struct_dir = os.path.join(base_dir, struct)
    if not os.path.isdir(struct_dir):
        print(f"[WARN] missing dir {struct_dir}")
        continue

    files = [f for f in os.listdir(struct_dir) if f.endswith(".graph")]
    valid_count = 0
    invalid_files = []

    for f in files:
        fpath = os.path.join(struct_dir, f)
        node_labels, edges = parse_graph(fpath)
        nodes = set(node_labels.keys())
        if not nodes:
            invalid_files.append((f, "没有节点信息"))
            os.remove(fpath)
            continue

        # 检查节点编号是否从0开始且连续
        ids_ok, reason_ids = check_node_ids(node_labels)
        if not ids_ok:
            invalid_files.append((f, reason_ids))
            os.remove(fpath)
            continue

        if not is_connected(nodes, edges):
            invalid_files.append((f, "图不连通"))
            os.remove(fpath)
            continue

        ok, reason = check_constraints(node_labels, edges)
        if not ok:
            invalid_files.append((f, reason))
            os.remove(fpath)
            continue

        # 生成规范化 graph 并覆盖原文件
        canon_str = canonicalize_graph(node_labels, edges)
        with open(fpath, "w") as fo:
            fo.write(canon_str)

        valid_count += 1

    print(f"[INFO] {struct.upper():6} 共有 {len(files)} 个文件，其中满足约束的查询图 {valid_count} 个")
    if invalid_files:
        print(f"[INFO] 不满足约束的查询图 {len(invalid_files)} 个，已删除：")
        for fname, reason in invalid_files:
            print(f"  - {fname}: {reason}")
    print()


我的数据图的一些属性如下：graphlib格式，边没标签，顶点有标签0，1，2，表示user\post\comment，每个用户可以连接多个post，多个comment，post可以连接多个用户，多个comment，但一个comment可以连接多个comment，只能连接一个用户和一个post，帮我基于随机游走的方式生成节点数量为4，6，8的稀疏查询，和稠密查询，每种至少N=25个，总共生成150个查询，且查询图的结构不能等价。查询图的结构如上，顶点第一列id从0开始，第二列是标签，第三列是度数 dataset name：dataset_three dataset features： edge no label，vertex has label，three class vertex：0-user\1-post\2-comment Vertex quantity：175688 Edge quantity：377658 query graph type: dense & sparse query graph vertex quantity: 4 ,6 ,8 query graph total num: 150 query graph generate method: BFS Extend inference predicate num : 1 inference node label: 1-post

#### 2. 生成推理节点文件infer_node.txt：
一个查询图的格式如下，三个顶点按照id依次编号为u0,u1,u2,接下来我想读取/home/wangshuo/resource/datasets/parler_data/dataset_one/query_graph文件夹下的所有查询文件，将标签为1的顶点所对应的标号保存到infer_node.txt，以下面查询为例要保存u1到文件中，
                                            t 3 3
                                            v 0 0 2
                                            v 1 1 2
                                            v 2 2 2
                                            e 1 2
                                            e 2 0
                                            e 1 0